In [1]:
import pandas as pd
import numpy as np
import re
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

print("=" * 80)
print("ColBERT-PRF Query Drift Analysis")
print("Analyzing topic drift through user selection patterns")
print("=" * 80)
print()

ColBERT-PRF Query Drift Analysis
Analyzing topic drift through user selection patterns



In [2]:
# ============================================================================
# SQL解析函数
# ============================================================================

def parse_sql_insert_statements(sql_file_path, table_name='logs'):
    """
    解析SQL文件中的INSERT语句，提取数据
    """
    print(f"正在解析SQL文件: {sql_file_path}")
    print("这可能需要几分钟时间，请耐心等待...")
    
    data = []
    
    with open(sql_file_path, 'r', encoding='utf-8') as f:
        buffer = ""
        insert_pattern = f"INSERT INTO `{table_name}` VALUES "
        
        for line_num, line in enumerate(f, 1):
            if line_num % 100000 == 0:
                print(f"  已处理 {line_num} 行...")
            
            # 跳过注释和空行
            if line.startswith('--') or line.startswith('/*') or line.startswith('*/') or not line.strip():
                continue
            
            buffer += line
            
            # 查找INSERT语句
            if insert_pattern in buffer:
                # 提取VALUES后面的内容
                try:
                    values_start = buffer.find(insert_pattern) + len(insert_pattern)
                    values_section = buffer[values_start:]
                    
                    # 使用正则表达式提取所有的值元组
                    pattern = r'\(([^)]+)\)'
                    matches = re.findall(pattern, values_section)
                    
                    for match in matches:
                        # 分割每个值
                        values = []
                        current = ""
                        in_quotes = False
                        quote_char = None
                        
                        for char in match:
                            if char in ('"', "'") and (not in_quotes or char == quote_char):
                                if in_quotes and char == quote_char:
                                    in_quotes = False
                                    quote_char = None
                                elif not in_quotes:
                                    in_quotes = True
                                    quote_char = char
                                current += char
                            elif char == ',' and not in_quotes:
                                values.append(current.strip().strip('"').strip("'"))
                                current = ""
                            else:
                                current += char
                        
                        if current:
                            values.append(current.strip().strip('"').strip("'"))
                        
                        if len(values) == 10:  # logs表有10列
                            data.append(values)
                    
                    buffer = ""
                    
                except Exception as e:
                    print(f"警告: 第{line_num}行解析出错: {e}")
                    buffer = ""
                    continue
    
    print(f"✓ SQL文件解析完成，共提取 {len(data)} 条记录")
    
    # 创建DataFrame
    columns = ['id', 'user_id', 'qid', 'docno', 'event_type', 
               'start_idx', 'end_idx', 'duration', 'pass_flag', 'timestamp']
    df = pd.DataFrame(data, columns=columns)
    
    # 转换数据类型
    df['id'] = pd.to_numeric(df['id'], errors='coerce')
    df['qid'] = pd.to_numeric(df['qid'], errors='coerce')
    df['start_idx'] = pd.to_numeric(df['start_idx'], errors='coerce')
    df['end_idx'] = pd.to_numeric(df['end_idx'], errors='coerce')
    df['duration'] = pd.to_numeric(df['duration'], errors='coerce')
    df['pass_flag'] = pd.to_numeric(df['pass_flag'], errors='coerce')
    df['timestamp'] = pd.to_datetime(df['timestamp'], errors='coerce')
    
    return df

In [3]:
# ============================================================================
# 第一步：加载数据
# ============================================================================

print("Loading data...")
print()

# 读取logs数据 - 支持两种方式
# 方式1：从SQL文件直接解析
# 方式2：从CSV文件读取

USE_SQL = True  # 设置为True从SQL解析，False从CSV读取

if USE_SQL:
    try:
        sql_file = 'backup_qe_v2_full.sql'
        logs_df = parse_sql_insert_statements(sql_file, table_name='logs')
        print(f"✓ Logs loaded from SQL: {len(logs_df)} records")
    except FileNotFoundError:
        print(f"✗ {sql_file} not found. Trying CSV instead...")
        USE_SQL = False

if not USE_SQL:
    try:
        logs_df = pd.read_csv('logs.csv')
        print(f"✓ Logs loaded from CSV: {len(logs_df)} records")
    except FileNotFoundError:
        print("✗ Neither SQL file nor logs.csv found. Please prepare data first.")
        exit(1)

# 读取interleaving结果
try:
    interleaving_df = pd.read_csv('result_preference_based_with_text_gold.csv')
    print(f"✓ Interleaving results loaded: {len(interleaving_df)} records")
except FileNotFoundError:
    print("✗ result_preference_based_with_text_gold.csv")
    exit(1)

print()

Loading data...

正在解析SQL文件: backup_qe_v2_full.sql
这可能需要几分钟时间，请耐心等待...
✓ SQL文件解析完成，共提取 48881 条记录
✓ Logs loaded from SQL: 48881 records
✓ Interleaving results loaded: 387 records



In [4]:
# ============================================================================
# 第二步：筛选目标用户
# ============================================================================

print("=" * 80)
print("User Filtering")
print("=" * 80)
print()

# 方法1：手动指定27个用户ID（如果你已经知道）
MANUAL_USERS = ['5a9fa90f873cda00010d29d8',
                '67e85a8becc5f7048616c4be',
                '679153802a458f825959cf4a',
                '5a0c4184fe645f0001e9f5dd',
                '5d31dc6e42678e001a0bdedd',
                '5589b65afdf99b118adfa595',
                '664dad1672f2f70e4aa2a012',
                '6601a50336e8fbd902820049',
                '5c4d8d706b23ab0001a34af2',
                '5e3a9e0dfbce8a296258c297',
                '581ccd016c73180001fa5b2a',
                '677bccba65784d9a5a876468',
                '662e3e72d494903379b6956a',
                '6637b21240220c2517fffa1a',
                '6626aca000aab07fdd4147c0',
                '65f860a1ab03181926bce465',
                '660453c9877e571d12b555f7',
                '5d31dc6e42678e001a0bdedd',
                '60b542b8fbb1a6a2678f5054',
                '5ce42b3454e33d001918923a',
                '63106ea4e8fe9c6cd32bf358',
                '5a70b6b882968f0001a6b12f',
                '67f2fbc77b115d1edb94ef11',
                '66afcaa60f7d8f58dc21db8e',
                '667b24c6c25f7a12d08d9ab6',
                '66ac261efa487682e38c3024',
                '65e4a6bdde84e615dcac2651',
                '67842c60c9cba1e154a69bbd',
                '63f0ac76a8c1914f6f163539',
                '66f5617d6ea358aa9f275019',
                '5c7ceda01d2afc0001f4ad1d',
                '62fb8e94ce94179d6a4e8277',
                '6641059cdec71f7b7d5a5e2b',
                '665a222ec21c09f2b8731fde',
                '63ea4d64095b3286433ed52e',
                '5b490405bc06c90001f8d6a5',
                '671d39090c9e2ad9ec75a03c',
                '66db9db4324609f1a7231f49',
                '5ebaa6dd28b52b000c94b78c',
                '5f88877885e51d02acd71b12',
                '5ef89ca507b0e3040299f1a3',
                '66aa95c94ab0bbc1be429c3e',
                '64480b662313e1b5ec1c7d57',
                '5d37c0d61b03b5000151950b',
                '6100372e3a116cb5cb36ebea',
                '60ac66a8e1bf5a1c51fa864c',
                '60719f4b18e2b30c9e5a562d',
                '5fc98fe1b105424d4b03381b',
                '67ef32ae3b9945398e55b406',
                '5a9e9fc46219a30001f54994',
                '5d0dca2cecd8f60017eaddb3',
                '66019f5e10adbe524f2a844a',
                '5df7b6b6ef7f5e55506510c7',
                '5f53b958c8cfea6e2104c5b6']

# 方法2：自动选择（如果列表为空，设置为[]使用所有用户）
if len(MANUAL_USERS) == 0:
    print("No manual user list provided. Using all users in the dataset.")
    target_users = logs_df['user_id'].unique().tolist()
else:
    target_users = MANUAL_USERS
    print(f"Using manually specified user list: {len(target_users)} users")

print(f"Target users to analyze: {len(target_users)}")
print()

# 筛选目标用户的日志
print("Filtering logs for target users...")
original_count = len(logs_df)
logs_df = logs_df[logs_df['user_id'].isin(target_users)].copy()
filtered_count = len(logs_df)

print(f"✓ Filtered from {original_count} to {filtered_count} records")
print(f"✓ Logs from {logs_df['user_id'].nunique()} unique users")

if logs_df['user_id'].nunique() < len(target_users):
    missing_users = set(target_users) - set(logs_df['user_id'].unique())
    print(f"⚠️  Warning: {len(missing_users)} users have no logs in the dataset")
    if len(missing_users) <= 5:
        print(f"   Missing users: {list(missing_users)[:5]}")

print()


User Filtering

Using manually specified user list: 54 users
Target users to analyze: 54

Filtering logs for target users...
✓ Filtered from 48881 to 14841 records
✓ Logs from 53 unique users
⚠️  Warning: 0 users have no logs in the dataset
   Missing users: []



In [5]:
# ============================================================================
# 第二步之二：时间筛选（过滤旧实验数据）⭐ 重要
# ============================================================================

print("Filtering by timestamp to remove old experiment data...")

# 方法1：指定当前实验的开始日期
EXPERIMENT_START_DATE = '2025-11-01'  # 修改为你的实验开始日期

# 方法2：或者直接设置为 None 跳过时间筛选
# EXPERIMENT_START_DATE = None

if EXPERIMENT_START_DATE is not None:
    # 转换为datetime
    logs_df['timestamp'] = pd.to_datetime(logs_df['timestamp'], errors='coerce')
    experiment_start = pd.to_datetime(EXPERIMENT_START_DATE)
    
    before_time_filter = len(logs_df)
    logs_df = logs_df[logs_df['timestamp'] >= experiment_start].copy()
    after_time_filter = len(logs_df)
    
    removed_count = before_time_filter - after_time_filter
    print(f"✓ Removed {removed_count} records before {EXPERIMENT_START_DATE}")
    print(f"✓ Remaining records: {after_time_filter}")
    
    # 显示时间范围
    if len(logs_df) > 0:
        min_time = logs_df['timestamp'].min()
        max_time = logs_df['timestamp'].max()
        print(f"✓ Data time range: {min_time} to {max_time}")
else:
    print("⚠️  Time filtering disabled. Using all historical data.")
    print("   This may include data from previous experiments!")

print()

Filtering by timestamp to remove old experiment data...
✓ Removed 0 records before 2025-11-01
✓ Remaining records: 14841
✓ Data time range: 2025-11-26 12:23:28 to 2025-12-01 20:34:08



In [7]:
# ============================================================================
# 第三步：数据预处理和标签映射
# ============================================================================

print("Data preprocessing...")
print()

# 显示origin_label的分布
print("Origin Label Distribution:")
label_counts = interleaving_df['origin_label'].value_counts()
print(label_counts)
print()

# 创建文档标签映射字典 (qid, docno) -> origin_label
doc_label_map = {}
for _, row in interleaving_df.iterrows():
    key = (str(row['qid']), str(row['docno']))
    doc_label_map[key] = row['origin_label']

print(f"✓ Created document label mapping: {len(doc_label_map)} documents")
print()

# 统计每种标签的文档数量
label_doc_counts = {
    'E2E-Only': len(interleaving_df[interleaving_df['origin_label'] == 'E2E-Only']),
    'PRF-Only': len(interleaving_df[interleaving_df['origin_label'] == 'PRF-Only']),
    'Both': len(interleaving_df[interleaving_df['origin_label'] == 'Both']),
    'A-Only': len(interleaving_df[interleaving_df['origin_label'] == 'A-Only']),
    'B-Only': len(interleaving_df[interleaving_df['origin_label'] == 'B-Only']),
    'Easy-Negative': len(interleaving_df[interleaving_df['origin_label'] == 'Easy-Negative'])
}

print("Document counts by label:")
for label, count in label_doc_counts.items():
    print(f"  {label}: {count} documents")
print()

Data preprocessing...

Origin Label Distribution:
origin_label
E2E-Only         99
Both             94
PRF-Only         81
A-Only           51
B-Only           51
Easy-Negative    11
Name: count, dtype: int64

✓ Created document label mapping: 387 documents

Document counts by label:
  E2E-Only: 99 documents
  PRF-Only: 81 documents
  Both: 94 documents
  A-Only: 51 documents
  B-Only: 51 documents
  Easy-Negative: 11 documents



In [8]:
# ============================================================================
# 第四步：分析所有事件统计
# ============================================================================

print("=" * 80)
print("Overall Statistics")
print("=" * 80)
print()

# 1. 总打开文档数量
open_doc_logs = logs_df[logs_df['event_type'] == 'OPEN_DOC']
total_open_docs = len(open_doc_logs)

# 2. 总passage选择数量（原始）
passage_selection_logs = logs_df[logs_df['event_type'] == 'PASSAGE_SELECTION']
total_raw_passages = len(passage_selection_logs)

# 3. 总取消数量（详细分类）
cancel_logs = logs_df[logs_df['event_type'] == 'CANCEL_DOC']
total_cancels = len(cancel_logs)

# 3a. Cancel without selection
cancel_without_selection = cancel_logs[
    (cancel_logs['start_idx'] == -1) & (cancel_logs['end_idx'] == -1)
]
total_cancel_without_selection = len(cancel_without_selection)

# 3b. Cancel after selection
cancel_after_selection = cancel_logs[
    (cancel_logs['start_idx'] != -1) | (cancel_logs['end_idx'] != -1)
]
total_cancel_after_selection = len(cancel_after_selection)

# 4. 取消去掉后的总passage选择数量
total_final_passages = total_raw_passages - total_cancel_after_selection

print(f"1. Total Documents Opened: {total_open_docs}")
print(f"2. Total Passage Selections (raw): {total_raw_passages}")
print(f"3. Total Cancellations: {total_cancels}")
print(f"   - Cancel without selection: {total_cancel_without_selection}")
print(f"   - Cancel after selection: {total_cancel_after_selection}")
print(f"4. Total Passage Selections (final): {total_final_passages}")
print()

Overall Statistics

1. Total Documents Opened: 6722
2. Total Passage Selections (raw): 6104
3. Total Cancellations: 650
   - Cancel without selection: 650
   - Cancel after selection: 0
4. Total Passage Selections (final): 6104



In [9]:
# ============================================================================
# 第五步：按origin_label统计passage选择
# ============================================================================

print("=" * 80)
print("Passage Selection by Origin Label")
print("=" * 80)
print()

# 为每个passage selection事件标注origin_label
passage_selections_with_label = []

for _, log in passage_selection_logs.iterrows():
    qid_str = str(int(log['qid'])) if pd.notna(log['qid']) else '0'
    docno_str = str(log['docno'])
    key = (qid_str, docno_str)
    
    label = doc_label_map.get(key, 'Unknown')
    
    passage_selections_with_label.append({
        'user_id': log['user_id'],
        'qid': qid_str,
        'docno': docno_str,
        'origin_label': label,
        'start_idx': log['start_idx'],
        'end_idx': log['end_idx'],
        'duration': log['duration']
    })

passage_df = pd.DataFrame(passage_selections_with_label)

# 统计每种标签的选择数量
label_selection_counts = passage_df['origin_label'].value_counts()

print("Passage selections by origin label (raw):")
for label in ['E2E-Only','PRF-Only','Both', 'A-Only', 'B-Only', 'Easy-Negative', 'Unknown']:
    count = label_selection_counts.get(label, 0)
    percentage = (count / total_raw_passages * 100) if total_raw_passages > 0 else 0
    
    # 计算每个文档的平均选择数
    doc_count = label_doc_counts.get(label, 0)
    if label == 'Unknown':
        doc_count = 1  # 避免除以0
    avg_per_doc = count / doc_count if doc_count > 0 else 0
    
    print(f"\n{label}:")
    print(f"  Total selections: {count}")
    print(f"  Percentage: {percentage:.2f}%")
    print(f"  Document count: {doc_count}")
    print(f"  Avg selections per document: {avg_per_doc:.2f}")

print()

# 处理cancel after selection，标注这些取消的origin_label
cancelled_passages_with_label = []

for _, log in cancel_after_selection.iterrows():
    qid_str = str(int(log['qid'])) if pd.notna(log['qid']) else '0'
    docno_str = str(log['docno'])
    key = (qid_str, docno_str)
    
    label = doc_label_map.get(key, 'Unknown')
    cancelled_passages_with_label.append({
        'qid': qid_str,
        'docno': docno_str,
        'origin_label': label
    })

cancelled_df = pd.DataFrame(cancelled_passages_with_label)

# 统计被取消的passage的label分布
if len(cancelled_df) > 0:
    print("Cancelled passages by origin label:")
    cancelled_label_counts = cancelled_df['origin_label'].value_counts()
    for label in ['E2E-Only','PRF-Only','Both', 'A-Only', 'B-Only', 'Easy-Negative', 'Unknown']:
        count = cancelled_label_counts.get(label, 0)
        print(f"  {label}: {count}")
    print()

# 计算最终的passage选择（减去取消的）
final_label_counts = {}
for label in ['E2E-Only','PRF-Only','Both', 'A-Only', 'B-Only', 'Easy-Negative', 'Unknown']:
    raw_count = label_selection_counts.get(label, 0)
    cancelled_count = cancelled_label_counts.get(label, 0) if len(cancelled_df) > 0 else 0
    final_count = raw_count - cancelled_count
    final_label_counts[label] = final_count

print("Passage selections by origin label (final, after removing cancellations):")
for label in ['E2E-Only','PRF-Only','Both', 'A-Only', 'B-Only', 'Easy-Negative', 'Unknown']:
    count = final_label_counts[label]
    percentage = (count / total_final_passages * 100) if total_final_passages > 0 else 0
    
    doc_count = label_doc_counts.get(label, 0)
    if label == 'Unknown':
        doc_count = 1
    avg_per_doc = count / doc_count if doc_count > 0 else 0
    
    print(f"\n{label}:")
    print(f"  Final selections: {count}")
    print(f"  Percentage: {percentage:.2f}%")
    print(f"  Document count: {doc_count}")
    print(f"  Avg selections per document: {avg_per_doc:.2f}")

print()

Passage Selection by Origin Label

Passage selections by origin label (raw):

E2E-Only:
  Total selections: 1314
  Percentage: 21.53%
  Document count: 99
  Avg selections per document: 13.27

PRF-Only:
  Total selections: 1253
  Percentage: 20.53%
  Document count: 81
  Avg selections per document: 15.47

Both:
  Total selections: 1755
  Percentage: 28.75%
  Document count: 94
  Avg selections per document: 18.67

A-Only:
  Total selections: 863
  Percentage: 14.14%
  Document count: 51
  Avg selections per document: 16.92

B-Only:
  Total selections: 868
  Percentage: 14.22%
  Document count: 51
  Avg selections per document: 17.02

Easy-Negative:
  Total selections: 51
  Percentage: 0.84%
  Document count: 11
  Avg selections per document: 4.64

Unknown:
  Total selections: 0
  Percentage: 0.00%
  Document count: 1
  Avg selections per document: 0.00

Passage selections by origin label (final, after removing cancellations):

E2E-Only:
  Final selections: 1314
  Percentage: 21.53%
  

In [15]:
# ============================================================================
# 查询级别的系统偏好分析
# 目标:识别哪些查询受益于PRF,哪些受挫于PRF,哪些无所谓
# ============================================================================

print("=" * 80)
print("Query-Level System Preference Analysis")
print("=" * 80)
print()

# 获取所有查询ID
all_qids = passage_df['qid'].unique()

# 两个结果列表
interleaving_results = []  # Interleaving测试结果
known_label_results = []   # 已知标签的点击统计

for qid in all_qids:
    query_passages = passage_df[passage_df['qid'] == qid]
    
    # 统计各种标签的点击
    e2e_only_clicks = len(query_passages[query_passages['origin_label'] == 'E2E-Only'])
    prf_only_clicks = len(query_passages[query_passages['origin_label'] == 'PRF-Only'])
    a_only_clicks = len(query_passages[query_passages['origin_label'] == 'A-Only'])
    b_only_clicks = len(query_passages[query_passages['origin_label'] == 'B-Only'])
    both_clicks = len(query_passages[query_passages['origin_label'] == 'Both'])
    
    # 统计该查询对应的各标签文档数量
    query_interleaving = interleaving_df[interleaving_df['qid'] == int(qid)]
    e2e_only_docs = len(query_interleaving[query_interleaving['origin_label'] == 'E2E-Only'])
    prf_only_docs = len(query_interleaving[query_interleaving['origin_label'] == 'PRF-Only'])
    a_only_docs = len(query_interleaving[query_interleaving['origin_label'] == 'A-Only'])
    b_only_docs = len(query_interleaving[query_interleaving['origin_label'] == 'B-Only'])
    both_docs = len(query_interleaving[query_interleaving['origin_label'] == 'Both'])
    
    total_clicks = e2e_only_clicks + prf_only_clicks + a_only_clicks + b_only_clicks + both_clicks
    
    if total_clicks == 0:
        continue
    
    # 表格1: Interleaving测试结果 (A-Only, B-Only, Both)
    # 只有当至少有一个interleaving文档(a_only, b_only, both)时才添加
    interleaving_total = a_only_clicks + b_only_clicks + both_clicks
    
    if a_only_docs > 0 or b_only_docs > 0 or both_docs > 0:
        discriminative_clicks = a_only_clicks + b_only_clicks
        
        if discriminative_clicks > 0:
            b_preference_ratio = b_only_clicks / discriminative_clicks
            
            if b_preference_ratio >= 0.6:
                preference = "PRF-Benefit"
            elif b_preference_ratio <= 0.4:
                preference = "PRF-Hurt"
            else:
                preference = "Neutral"
        else:
            preference = "Insufficient-Data"
            b_preference_ratio = None
        
        interleaving_results.append({
            'qid': qid,
            'a_only_clicks': f"{a_only_clicks}({a_only_docs})",
            'b_only_clicks': f"{b_only_clicks}({b_only_docs})",
            'both_clicks': f"{both_clicks}({both_docs})",
            'total_clicks': interleaving_total,
            'discriminative_clicks': discriminative_clicks,
            'b_preference_ratio': b_preference_ratio,
            'preference': preference
        })
    
    # 表格2: 已知标签的点击统计 (E2E-Only, PRF-Only)
    # 只有当该查询至少有一个E2E-Only或PRF-Only文档时才添加
    if e2e_only_docs > 0 or prf_only_docs > 0:
        known_label_results.append({
            'qid': qid,
            'e2e_only_clicks': f"{e2e_only_clicks}({e2e_only_docs})",
            'prf_only_clicks': f"{prf_only_clicks}({prf_only_docs})",
            'total_known_clicks': e2e_only_clicks + prf_only_clicks
        })

# 转换为DataFrame
interleaving_df_result = pd.DataFrame(interleaving_results)
known_label_df_result = pd.DataFrame(known_label_results)

# 保存到CSV
interleaving_df_result.to_csv('preference_interleaving.csv', index=False)
known_label_df_result.to_csv('preference_known_labels.csv', index=False)

print("\n" + "=" * 80)
print("Table 1: Interleaving Test Results (A-Only vs B-Only vs Both)")
print("=" * 80)
display(interleaving_df_result)

print("\n" + "=" * 80)
print("Table 2: Known Label Clicks (E2E-Only vs PRF-Only)")
print("=" * 80)
display(known_label_df_result)

Query-Level System Preference Analysis


Table 1: Interleaving Test Results (A-Only vs B-Only vs Both)


,qid,a_only_clicks,b_only_clicks,both_clicks,total_clicks,discriminative_clicks,b_preference_ratio,preference
0,47923,15(1),8(1),112(6),135,23,0.347826,PRF-Hurt
1,87181,47(3),45(3),62(3),154,92,0.489130,Neutral
2,156493,22(1),17(1),116(6),155,39,0.435897,Neutral
3,168216,36(2),33(2),101(5),170,69,0.478261,Neutral
4,182539,22(2),22(2),52(5),96,44,0.500000,Neutral
5,183378,54(3),44(3),40(2),138,98,0.448980,Neutral
6,207786,17(3),29(3),38(3),84,46,0.630435,PRF-Benefit
7,264014,62(3),54(3),43(2),159,116,0.465517,Neutral
8,490595,47(3),49(3),62(3),158,96,0.510417,Neutral
9,527433,16(2),11(2),63(5),90,27,0.407407,Neutral



Table 2: Known Label Clicks (E2E-Only vs PRF-Only)


,qid,e2e_only_clicks,prf_only_clicks,total_known_clicks
0,19335,0(0),122(9),122
1,87452,60(9),0(0),60
2,104861,0(0),148(9),148
3,130510,0(0),156(9),156
4,131843,0(0),132(9),132
5,146187,120(9),0(0),120
6,148538,0(0),144(9),144
7,359349,141(9),0(0),141
8,405717,0(0),107(9),107
9,443396,0(0),99(9),99


In [10]:
# ============================================================================
# 输出统计结果
# ============================================================================

print(f"Total queries analyzed: {len(preference_df)}")
print()
print("Query Distribution by PRF Effect:")
print("-" * 80)

for pref in ['PRF-Benefit', 'PRF-Hurt', 'Neutral', 'Insufficient-Data']:
    subset = preference_df[preference_df['preference'] == pref]
    count = len(subset)
    percentage = count / len(preference_df) * 100
    
    print(f"\n{pref}:")
    print(f"  Count: {count} queries ({percentage:.2f}%)")
    
    if pref != 'Insufficient-Data' and count > 0:
        avg_ratio = subset['b_preference_ratio'].mean()
        avg_disc_clicks = subset['discriminative_clicks'].mean()
        print(f"  Average B-preference ratio: {avg_ratio:.3f}")
        print(f"  Average discriminative clicks: {avg_disc_clicks:.1f}")

Total queries analyzed: 43

Query Distribution by PRF Effect:
--------------------------------------------------------------------------------

PRF-Benefit:
  Count: 9 queries (20.93%)
  Average B-preference ratio: 0.779
  Average discriminative clicks: 47.1

PRF-Hurt:
  Count: 11 queries (25.58%)
  Average B-preference ratio: 0.280
  Average discriminative clicks: 36.3

Neutral:
  Count: 21 queries (48.84%)
  Average B-preference ratio: 0.500
  Average discriminative clicks: 64.9

Insufficient-Data:
  Count: 2 queries (4.65%)


In [18]:
# ============================================================================
# User Study 1 vs User Study 2 对比分析
# 目标：验证Selective PRF策略是否有效
# ============================================================================

print("=" * 80)
print("Selective PRF Strategy Validation")
print("User Study 1 (Interleaving) vs User Study 2 (Selective)")
print("=" * 80)
print()

# 读取CSV文件
study1_df = pd.read_csv('preference.csv')
study2_interleaving_df = pd.read_csv('preference_interleaving.csv')
study2_known_df = pd.read_csv('preference_known_labels.csv')

print(f"User Study 1 queries: {len(study1_df)}")
print(f"User Study 2 interleaving queries: {len(study2_interleaving_df)}")
print(f"User Study 2 known label queries: {len(study2_known_df)}")
print()

# 合并分析
comparison_results = []

# 遍历Study 1的所有查询
for _, study1_row in study1_df.iterrows():
    qid = study1_row['qid']
    
    # 在Study 2中查找对应的数据
    study2_interleaving = study2_interleaving_df[study2_interleaving_df['qid'] == qid]
    study2_known = study2_known_df[study2_known_df['qid'] == qid]
    
    # Study 1的信息
    preference = study1_row['preference']
    study1_total = study1_row['total_clicks']
    study1_a_only = study1_row['a_only_clicks']
    study1_b_only = study1_row['b_only_clicks']
    study1_both = study1_row['both_clicks']
    
    # Study 2的信息
    study2_interleaving_total = study2_interleaving.iloc[0]['total_clicks'] if len(study2_interleaving) > 0 else 0
    study2_known_total = study2_known.iloc[0]['total_known_clicks'] if len(study2_known) > 0 else 0
    study2_total = study2_interleaving_total + study2_known_total
    
    # Study 2 interleaving部分
    study2_a_only = study2_interleaving.iloc[0]['a_only_clicks'] if len(study2_interleaving) > 0 else "0(0)"
    study2_b_only = study2_interleaving.iloc[0]['b_only_clicks'] if len(study2_interleaving) > 0 else "0(0)"
    study2_both = study2_interleaving.iloc[0]['both_clicks'] if len(study2_interleaving) > 0 else "0(0)"
    
    # Study 2 known label部分
    study2_e2e = study2_known.iloc[0]['e2e_only_clicks'] if len(study2_known) > 0 else "0(0)"
    study2_prf = study2_known.iloc[0]['prf_only_clicks'] if len(study2_known) > 0 else "0(0)"
    
    # 判断策略和解析文档数
    if len(study2_known) > 0:
        e2e_docs = int(study2_e2e.split('(')[1].split(')')[0])
        prf_docs = int(study2_prf.split('(')[1].split(')')[0])
        
        if preference == "PRF-Hurt":
            strategy = "Use E2E-Only"
            strategy_correct = (prf_docs == 0 and e2e_docs > 0)
        elif preference == "PRF-Benefit":
            strategy = "Use PRF-Only"
            strategy_correct = (e2e_docs == 0 and prf_docs > 0)
        elif preference == "Neutral":
            strategy = "Mixed or Either"
            strategy_correct = True
        else:
            strategy = "Unknown"
            strategy_correct = None
    else:
        strategy = "Interleaving Only"
        strategy_correct = None
    
    # 计算点击增益
    click_gain = study2_total - study1_total
    click_gain_pct = (click_gain / study1_total * 100) if study1_total > 0 else 0
    
    # 判断结果是否符合预期
    if preference in ["PRF-Hurt", "PRF-Benefit"]:
        outcome_match = (click_gain > 0)
    elif preference == "Neutral":
        outcome_match = (abs(click_gain_pct) < 10)
    else:
        outcome_match = None
    
    comparison_results.append({
        'qid': qid,
        # Study 1
        'study1_preference': preference,
        'study1_total': study1_total,
        'study1_a_only': study1_a_only,
        'study1_b_only': study1_b_only,
        'study1_both': study1_both,
        # Study 2
        'study2_strategy': strategy,
        'study2_total': study2_total,
        'study2_e2e': study2_e2e,
        'study2_prf': study2_prf,
        'study2_a_only': study2_a_only,
        'study2_b_only': study2_b_only,
        'study2_both': study2_both,
        # 对比
        'click_gain': click_gain,
        'click_gain_pct': f"{click_gain_pct:.1f}%",
        'strategy_correct': strategy_correct,
        'outcome_match': outcome_match
    })

# 转换为DataFrame
comparison_df = pd.DataFrame(comparison_results)
comparison_df.to_csv('selective_prf_validation.csv', index=False)

print("\nComparison Table:")
print("=" * 80)
display(comparison_df)

# 统计分析
print("\n" + "=" * 80)
print("Summary Statistics")
print("=" * 80)

for pref in ['PRF-Hurt', 'PRF-Benefit', 'Neutral']:
    subset = comparison_df[comparison_df['study1_preference'] == pref]
    if len(subset) == 0:
        continue
    
    print(f"\n{pref} Queries ({len(subset)} queries):")
    print("-" * 80)
    
    avg_gain = subset['click_gain'].mean()
    
    if pref != 'Neutral':
        strategy_correct_count = subset['strategy_correct'].sum()
        strategy_correct_rate = strategy_correct_count / len(subset) * 100
        outcome_match_count = subset['outcome_match'].sum()
        outcome_match_rate = outcome_match_count / len(subset) * 100
        print(f"  Strategy Correctly Applied: {strategy_correct_count}/{len(subset)} ({strategy_correct_rate:.1f}%)")
        print(f"  Outcome Matched Expectation: {outcome_match_count}/{len(subset)} ({outcome_match_rate:.1f}%)")
    
    print(f"  Average Click Gain: {avg_gain:.1f}")
    print(f"  Queries with positive gain: {(subset['click_gain'] > 0).sum()} / {len(subset)}")
    print(f"  Queries with negative gain: {(subset['click_gain'] < 0).sum()} / {len(subset)}")
    print(f"  Queries with no change: {(subset['click_gain'] == 0).sum()} / {len(subset)}")

# 整体效果
print("\n" + "=" * 80)
print("Overall Selective PRF Effect")
print("=" * 80)
total_gain = comparison_df['click_gain'].sum()
avg_gain = comparison_df['click_gain'].mean()
positive_queries = (comparison_df['click_gain'] > 0).sum()
negative_queries = (comparison_df['click_gain'] < 0).sum()
zero_queries = (comparison_df['click_gain'] == 0).sum()

print(f"Total Click Gain: {total_gain}")
print(f"Average Click Gain per Query: {avg_gain:.2f}")
print(f"Queries Improved: {positive_queries} / {len(comparison_df)} ({positive_queries/len(comparison_df)*100:.1f}%)")
print(f"Queries Degraded: {negative_queries} / {len(comparison_df)} ({negative_queries/len(comparison_df)*100:.1f}%)")
print(f"Queries Unchanged: {zero_queries} / {len(comparison_df)} ({zero_queries/len(comparison_df)*100:.1f}%)")

# 显著性检验
from scipy import stats
t_stat, p_value = stats.ttest_1samp(comparison_df['click_gain'], 0)
print(f"\nStatistical Test (paired t-test):")
print(f"  t-statistic: {t_stat:.3f}")
print(f"  p-value: {p_value:.4f}")
if p_value < 0.05:
    print(f"  Result: ✓ Statistically significant improvement (p < 0.05)")
else:
    print(f"  Result: ✗ No significant difference (p >= 0.05)")

Selective PRF Strategy Validation
User Study 1 (Interleaving) vs User Study 2 (Selective)

User Study 1 queries: 43
User Study 2 interleaving queries: 23
User Study 2 known label queries: 20


Comparison Table:


,qid,study1_preference,study1_total,study1_a_only,study1_b_only,study1_both,study2_strategy,study2_total,study2_e2e,study2_prf,study2_a_only,study2_b_only,study2_both,click_gain,click_gain_pct,strategy_correct,outcome_match
0,104861,PRF-Benefit,110,23(3),57(3),30(2),Use PRF-Only,148,0(0),148(9),0(0),0(0),0(0),38,34.5%,True,True
1,130510,PRF-Benefit,135,12(2),30(2),93(5),Use PRF-Only,156,0(0),156(9),0(0),0(0),0(0),21,15.6%,True,True
2,131843,PRF-Benefit,110,1(2),25(2),84(4),Use PRF-Only,132,0(0),132(9),0(0),0(0),0(0),22,20.0%,True,True
3,146187,PRF-Hurt,120,27(2),8(2),85(5),Use E2E-Only,120,120(9),0(0),0(0),0(0),0(0),0,0.0%,True,False
4,148538,PRF-Benefit,102,12(2),33(2),57(4),Use PRF-Only,144,0(0),144(9),0(0),0(0),0(0),42,41.2%,True,True
5,156493,Neutral,149,17(1),19(1),113(6),Interleaving Only,155,0(0),0(0),22(1),17(1),116(6),6,4.0%,None,True
6,1037798,PRF-Benefit,98,3(3),40(3),55(3),Use PRF-Only,166,0(0),166(9),0(0),0(0),0(0),68,69.4%,True,True
7,1063750,PRF-Hurt,56,20(3),6(3),30(2),Use E2E-Only,125,125(9),0(0),0(0),0(0),0(0),69,123.2%,True,True
8,1103812,PRF-Hurt,82,8(1),4(1),70(7),Use E2E-Only,116,116(9),0(0),0(0),0(0),0(0),34,41.5%,True,True
9,1106007,Neutral,144,30(2),40(2),74(4),Interleaving Only,195,0(0),0(0),42(2),52(2),101(4),51,35.4%,None,False



Summary Statistics

PRF-Hurt Queries (11 queries):
--------------------------------------------------------------------------------
  Strategy Correctly Applied: 11/11 (100.0%)
  Outcome Matched Expectation: True/11 (9.1%)
  Average Click Gain: 37.3
  Queries with positive gain: 10 / 11
  Queries with negative gain: 0 / 11
  Queries with no change: 1 / 11

PRF-Benefit Queries (9 queries):
--------------------------------------------------------------------------------
  Strategy Correctly Applied: 9/9 (100.0%)
  Outcome Matched Expectation: True/9 (11.1%)
  Average Click Gain: 36.7
  Queries with positive gain: 9 / 9
  Queries with negative gain: 0 / 9
  Queries with no change: 0 / 9

Neutral Queries (21 queries):
--------------------------------------------------------------------------------
  Average Click Gain: 26.5
  Queries with positive gain: 18 / 21
  Queries with negative gain: 3 / 21
  Queries with no change: 0 / 21

Overall Selective PRF Effect
Total Click Gain: 1360
Averag

In [19]:
# ============================================================================
# 公平性分析：控制用户参与度差异
# ============================================================================

print("=" * 80)
print("Fairness Analysis: Controlling for User Engagement")
print("=" * 80)
print()

# 方法1: 使用归一化点击率 (Clicks per Document)
# 因为Study 2可能文档数量不同，我们应该比较每个文档的平均点击数
print("Method 1: Clicks per Document (Normalized)")
print("-" * 80)

normalized_results = []

for _, row in comparison_df.iterrows():
    qid = row['qid']
    
    # Study 1: 计算文档数
    def extract_doc_count(click_str):
        if pd.isna(click_str) or click_str == "0(0)":
            return 0, 0
        parts = str(click_str).split('(')
        clicks = int(parts[0])
        docs = int(parts[1].rstrip(')'))
        return clicks, docs
    
    # Study 1
    s1_a_clicks, s1_a_docs = extract_doc_count(row['study1_a_only'])
    s1_b_clicks, s1_b_docs = extract_doc_count(row['study1_b_only'])
    s1_both_clicks, s1_both_docs = extract_doc_count(row['study1_both'])
    s1_total_docs = s1_a_docs + s1_b_docs + s1_both_docs
    s1_clicks_per_doc = row['study1_total'] / s1_total_docs if s1_total_docs > 0 else 0
    
    # Study 2
    s2_e2e_clicks, s2_e2e_docs = extract_doc_count(row['study2_e2e'])
    s2_prf_clicks, s2_prf_docs = extract_doc_count(row['study2_prf'])
    s2_a_clicks, s2_a_docs = extract_doc_count(row['study2_a_only'])
    s2_b_clicks, s2_b_docs = extract_doc_count(row['study2_b_only'])
    s2_both_clicks, s2_both_docs = extract_doc_count(row['study2_both'])
    s2_total_docs = s2_e2e_docs + s2_prf_docs + s2_a_docs + s2_b_docs + s2_both_docs
    s2_clicks_per_doc = row['study2_total'] / s2_total_docs if s2_total_docs > 0 else 0
    
    # 归一化增益
    normalized_gain = s2_clicks_per_doc - s1_clicks_per_doc
    normalized_gain_pct = (normalized_gain / s1_clicks_per_doc * 100) if s1_clicks_per_doc > 0 else 0
    
    normalized_results.append({
        'qid': qid,
        'preference': row['study1_preference'],
        'study1_docs': s1_total_docs,
        'study2_docs': s2_total_docs,
        'study1_clicks_per_doc': s1_clicks_per_doc,
        'study2_clicks_per_doc': s2_clicks_per_doc,
        'normalized_gain': normalized_gain,
        'normalized_gain_pct': normalized_gain_pct
    })

normalized_df = pd.DataFrame(normalized_results)

# 统计归一化结果
for pref in ['PRF-Hurt', 'PRF-Benefit', 'Neutral']:
    subset = normalized_df[normalized_df['preference'] == pref]
    if len(subset) == 0:
        continue
    
    print(f"\n{pref} Queries ({len(subset)} queries):")
    avg_s1_cpd = subset['study1_clicks_per_doc'].mean()
    avg_s2_cpd = subset['study2_clicks_per_doc'].mean()
    avg_gain = subset['normalized_gain'].mean()
    avg_gain_pct = subset['normalized_gain_pct'].mean()
    
    print(f"  Study 1 avg clicks/doc: {avg_s1_cpd:.2f}")
    print(f"  Study 2 avg clicks/doc: {avg_s2_cpd:.2f}")
    print(f"  Normalized gain: {avg_gain:.2f} ({avg_gain_pct:.1f}%)")
    print(f"  Queries improved: {(subset['normalized_gain'] > 0).sum()} / {len(subset)}")

# 整体归一化效果
print(f"\nOverall Normalized Effect:")
print(f"  Average normalized gain: {normalized_df['normalized_gain'].mean():.2f}")
t_stat, p_value = stats.ttest_1samp(normalized_df['normalized_gain'], 0)
print(f"  t-statistic: {t_stat:.3f}, p-value: {p_value:.4f}")

print("\n" + "=" * 80)

# 方法2: 控制组对比 - 使用Neutral查询作为基准
print("\nMethod 2: Using Neutral Queries as Baseline")
print("-" * 80)

# Neutral查询应该在两个study中表现相似（如果用户参与度相同）
neutral_subset = comparison_df[comparison_df['study1_preference'] == 'Neutral']
neutral_avg_gain = neutral_subset['click_gain'].mean()

print(f"Neutral queries average gain: {neutral_avg_gain:.1f}")
print(f"This represents the 'baseline' user engagement difference")
print()

# 调整PRF-Hurt和PRF-Benefit的增益，减去Neutral的基准增益
adjusted_results = []

for _, row in comparison_df.iterrows():
    adjusted_gain = row['click_gain'] - neutral_avg_gain
    adjusted_results.append({
        'qid': row['qid'],
        'preference': row['study1_preference'],
        'raw_gain': row['click_gain'],
        'adjusted_gain': adjusted_gain
    })

adjusted_df = pd.DataFrame(adjusted_results)

for pref in ['PRF-Hurt', 'PRF-Benefit']:
    subset = adjusted_df[adjusted_df['preference'] == pref]
    if len(subset) == 0:
        continue
    
    print(f"\n{pref} Queries ({len(subset)} queries):")
    print(f"  Raw average gain: {subset['raw_gain'].mean():.1f}")
    print(f"  Adjusted average gain (removing baseline): {subset['adjusted_gain'].mean():.1f}")
    print(f"  Queries with positive adjusted gain: {(subset['adjusted_gain'] > 0).sum()} / {len(subset)}")

t_stat, p_value = stats.ttest_1samp(
    adjusted_df[adjusted_df['preference'].isin(['PRF-Hurt', 'PRF-Benefit'])]['adjusted_gain'], 
    0
)
print(f"\nAdjusted t-test: t={t_stat:.3f}, p={p_value:.4f}")

print("\n" + "=" * 80)

# 方法3: 效应量分析 (Effect Size - Cohen's d)
print("\nMethod 3: Effect Size Analysis (Cohen's d)")
print("-" * 80)

def cohens_d(x):
    return x.mean() / x.std() if x.std() > 0 else 0

for pref in ['PRF-Hurt', 'PRF-Benefit', 'Neutral']:
    subset = comparison_df[comparison_df['study1_preference'] == pref]
    if len(subset) == 0:
        continue
    
    d = cohens_d(subset['click_gain'])
    
    # Cohen's d解释: 0.2=small, 0.5=medium, 0.8=large
    if abs(d) < 0.2:
        magnitude = "negligible"
    elif abs(d) < 0.5:
        magnitude = "small"
    elif abs(d) < 0.8:
        magnitude = "medium"
    else:
        magnitude = "large"
    
    print(f"\n{pref}: Cohen's d = {d:.3f} ({magnitude} effect)")

print("\n" + "=" * 80)

# 方法4: 查看每个查询的相对改进
print("\nMethod 4: Relative Improvement per Query")
print("-" * 80)

relative_results = []

for _, row in comparison_df.iterrows():
    relative_improvement = (row['click_gain'] / row['study1_total'] * 100) if row['study1_total'] > 0 else 0
    relative_results.append({
        'qid': row['qid'],
        'preference': row['study1_preference'],
        'relative_improvement': relative_improvement
    })

relative_df = pd.DataFrame(relative_results)

for pref in ['PRF-Hurt', 'PRF-Benefit', 'Neutral']:
    subset = relative_df[relative_df['preference'] == pref]
    if len(subset) == 0:
        continue
    
    print(f"\n{pref} Queries:")
    print(f"  Average relative improvement: {subset['relative_improvement'].mean():.1f}%")
    print(f"  Median relative improvement: {subset['relative_improvement'].median():.1f}%")

# 保存所有分析结果
normalized_df.to_csv('fairness_analysis_normalized.csv', index=False)
adjusted_df.to_csv('fairness_analysis_adjusted.csv', index=False)
relative_df.to_csv('fairness_analysis_relative.csv', index=False)

print("\n" + "=" * 80)
print("Conclusion:")
print("-" * 80)
print("If the normalized/adjusted results still show significant improvements,")
print("then Selective PRF is truly effective, not just due to user engagement.")

Fairness Analysis: Controlling for User Engagement

Method 1: Clicks per Document (Normalized)
--------------------------------------------------------------------------------

PRF-Hurt Queries (11 queries):
  Study 1 avg clicks/doc: 9.48
  Study 2 avg clicks/doc: 13.27
  Normalized gain: 3.79 (57.4%)
  Queries improved: 10 / 11

PRF-Benefit Queries (9 queries):
  Study 1 avg clicks/doc: 12.12
  Study 2 avg clicks/doc: 15.47
  Normalized gain: 3.34 (28.5%)
  Queries improved: 9 / 9

Neutral Queries (21 queries):
  Study 1 avg clicks/doc: 14.96
  Study 2 avg clicks/doc: 18.08
  Normalized gain: 3.12 (25.0%)
  Queries improved: 18 / 21

Overall Normalized Effect:
  Average normalized gain: 3.37
  t-statistic: 9.385, p-value: 0.0000


Method 2: Using Neutral Queries as Baseline
--------------------------------------------------------------------------------
Neutral queries average gain: 26.5
This represents the 'baseline' user engagement difference


PRF-Hurt Queries (11 queries):
  Raw a